# Job Finder Chatbot — MLflow Evaluation
ISA632 Group Project | Evaluation & Validation

Uses `mlflow.genai.evaluate()` (MLflow 3.0) with custom Guidelines scorers and built-in scorers
applied to 12 realistic user questions from our ChatTesting session.

In [ ]:
%pip install -qqq -U databricks-sdk databricks-langchain databricks-vectorsearch langsmith>=0.1.125 langchain==0.3.27 mlflow-skinny[databricks]>=3.4.0 
dbutils.library.restartPython()

In [ ]:
# Cell 1 — Setup
import os
import json
import mlflow
import mlflow.pyfunc
import pandas as pd
from mlflow.genai import evaluate
from mlflow.genai.scorers import Guidelines, RelevanceToQuery, Safety

os.environ["VS_INDEX_NAME"] = "isa632_7474656346303369.boopatt.jobindex"
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment("/Users/boopatt@miamioh.edu/JobFinder_Eval")
print("Setup complete.")

In [ ]:
# Cell 2 — Load the Registered Model
model_uri = "models:/isa632_7474656346303369.boopatt.getstarted_job_listings/1"
model = mlflow.pyfunc.load_model(model_uri)
print(f"Model loaded: {model_uri}")

In [ ]:
# Cell 3 — Define predict_fn
@mlflow.trace
def predict_fn(messages):
    """Wraps the job finder agent for MLflow evaluation."""
    out = model.predict({"input": messages})

    if isinstance(out, dict) and "output" in out:
        try:
            return out["output"][-1]["content"][0]["text"].strip()
        except Exception:
            pass
    if isinstance(out, list):
        try:
            return out[-1]["content"][0]["text"].strip()
        except Exception:
            pass
    if isinstance(out, str):
        return out.strip()
    return str(out).strip()

# Quick smoke test
test_response = predict_fn([{"content": "What data analyst roles are available?", "role": "user"}])
print("Smoke test response (first 200 chars):")
print(test_response[:200])

In [ ]:
# Cell 4 — Load Evaluation Dataset
eval_rows = []
with open("/Workspace/Users/boopatt@miamioh.edu/Project/eval_data.jsonl") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        record = json.loads(line)
        eval_rows.append({
            "inputs": record["inputs"],
            "expectations": record["expectations"]
        })

eval_df = pd.DataFrame(eval_rows)
print(f"Loaded {len(eval_df)} evaluation questions.")
display(eval_df)

In [ ]:
# Cell 5 — Define Custom Guidelines Scorers
job_relevant_scorer = Guidelines(
    name="job_relevant",
    guidelines="""The response must reference real job postings from the database.
It should include at least one of: a job title, company name, job description, salary range, or qualification requirement.
If the question is about images or unrelated topics (e.g., asking for a photo), the response should clearly state it cannot help — that is also acceptable.
Responses that are vague, generic, or fabricate job information without citing the database are NOT acceptable."""
)

filter_aware_scorer = Guidelines(
    name="filter_aware",
    guidelines="""When the user specifies a filter — such as a location (e.g., New York), experience level (e.g., beginner), 
skill requirement (e.g., Python only), or work mode (e.g., remote) — the response must acknowledge that constraint explicitly.
If no matching results exist for the filter, the response must say so rather than returning unfiltered results silently.
If the question contains no filter constraints, this criterion does not apply — score as passing."""
)

print("Custom scorers defined: job_relevant, filter_aware")

In [ ]:
# Cell 6 — Run Evaluation
with mlflow.start_run(run_name="jobfinder_eval_v1") as eval_run:
    eval_results = evaluate(
        data=eval_df,
        predict_fn=predict_fn,
        scorers=[
            job_relevant_scorer,
            filter_aware_scorer,
            RelevanceToQuery(),
            Safety()
        ]
    )

print(f"Evaluation complete. Run ID: {eval_run.info.run_id}")
display(eval_results.tables["eval_results"])

In [ ]:
# Cell 7 — Summarize Results
results_df = eval_results.tables["eval_results"]

score_cols = [
    "job_relevant/value",
    "filter_aware/value",
    "relevance_to_query/value",
    "safety/value"
]

def to_numeric(val):
    """Convert boolean/string scorer values to 0 or 1."""
    if isinstance(val, bool):
        return int(val)
    if isinstance(val, str):
        return 1 if val.lower() in ("yes", "true") else 0
    return int(bool(val))

print("=== Evaluation Summary ===")
print(f"Total questions evaluated: {len(results_df)}")
print()
for col in score_cols:
    if col in results_df.columns:
        mean_val = results_df[col].apply(to_numeric).mean()
        label = col.replace("/value", "").replace("_", " ").title()
        print(f"{label:30s}: {mean_val:.0%}")

print()
print("View full results in MLflow UI \u2192 Experiments \u2192 JobFinder_Eval")

## Why We Stopped at Evaluation — Decision Not to Fine-Tune

After reviewing the evaluation results, we explicitly considered whether to proceed with LLM fine-tuning (LoRA) as the next improvement step and decided against it. Here is the full reasoning.

---

### What the Evaluation Told Us

The 58% Filter Awareness score was the only meaningful failure signal. Drilling into those 5 failing questions revealed a consistent pattern: the database simply does not contain entry-level or beginner-level job postings. The model was not making errors — it was correctly retrieving the most relevant documents available and honestly flagging mismatches via the MISMATCH FORMAT in the system prompt. This is a **data coverage gap**, not a model capability gap.

The remaining three scores (Safety 100%, Job Relevant 92%, Relevance to Query 92%) confirmed that the base model combined with our RAG pipeline and prompt engineering was already performing well for the queries our dataset could support.

---

### Why Fine-Tuning Would Not Fix This

Fine-tuning with LoRA adapts a model's **behavior and response style** — it teaches the model *how* to answer, not *what* to know. Our problem was the opposite: the model's behavior (following rules, using MISMATCH FORMAT, citing retrieved facts) was already correct. What was missing was **knowledge** — specifically, entry-level job postings that do not exist in our Vector Search index.

Attempting to fine-tune on 30 PDFs of job listings would not resolve this because:

1. **Wrong tool for the problem.** RAG retrieves live, specific facts at inference time. Fine-tuning encodes a static snapshot into model weights. Job listings change frequently — a fine-tuned model would become stale immediately and could not reliably recall exact salary figures, qualifications, or company names from weights the way a retriever can from an indexed document.

2. **Insufficient data volume and structure.** Fine-tuning requires structured `instruction → input → expected output` pairs. Our 30 PDFs would need to be manually converted into hundreds of labeled Q&A examples first. Even the ISA632 lab example — which used a simple style-adaptation task — flagged its 40-example dataset as a *toy dataset not suitable for production*. A knowledge-grounded job finder would require thousands of high-quality pairs to see meaningful improvement.

3. **Model scale mismatch.** The fine-tuning lab used a 0.5B parameter model (Qwen2.5-0.5B-Instruct), which is small enough to train on a single GPU in minutes. Our deployed model is Databricks Llama 4 Maverick — a far larger model that would require significant GPU resources and training time, well beyond the scope and timeline of this project.

4. **The project spec confirms this.** The ISA632 project instructions list fine-tuning as an optional advanced component, recommended only for teams with a strong technical background, and explicitly note it is not expected for a functional prototype.

---

### What Would Actually Improve the System

The evaluation pointed to two targeted fixes that address the root cause without fine-tuning:

| Issue | Root Cause | Correct Fix |
|---|---|---|
| Filter Aware 58% | No entry-level postings in the index | Expand the dataset with junior/entry-level job listings |
| Multi-criteria filtering | Vector search is semantic, not structured | Add metadata fields (seniority, location, skills) and use hybrid search |
| Response truncation (Q4) | `max_tokens=500` is too low | Increase token limit in `agent.py` |

These are data engineering and retrieval improvements — they stay within the RAG architecture that is already working well.

---

### Conclusion

We stopped at evaluation because the evaluation did its job: it told us precisely what was wrong and why. The failure mode was a data gap, not a model gap, and fine-tuning is not the right remedy for a data gap. Expanding the RAG pipeline with richer, more diverse job postings is the correct next step — and one that keeps the system maintainable, up-to-date, and aligned with the retrieval-first architecture we built.

## Evaluation Results & Inference

The table below shows the final scores from the `jobfinder_eval_v1` MLflow run across 12 realistic user questions.

| Metric | Score | Interpretation |
|---|---|---|
| **Safety** | 100% | No harmful or inappropriate content in any response — expected for a job search domain |
| **Job Relevant** | 92% | 11 of 12 responses were grounded in real job postings from the database. The 1 failure (Q11 — *"most needed skill for beginner level"*) deflected without citing actionable job data |
| **Relevance to Query** | 92% | Matches job relevance — the agent addressed what was asked in nearly all cases |
| **Filter Aware** | 58% | 7 of 12 responses correctly handled user-specified filters. The 5 failures involved constraints the database could not satisfy: beginner/entry-level experience (Q2, Q11, Q12), multi-criteria filtering (Q2 — location + skill + level), and background-based matching (Q5 — MSBA roles) |

### Key Findings

**Strengths:**
- The chatbot is reliable and safe across all query types
- Factual lookups — salary ranges, company locations, company culture, managerial roles — scored 100%
- The improved system prompt (MISMATCH FORMAT + Rule 4) successfully prevented the agent from presenting mismatched results as correct answers

**Main Limitation — Filter Awareness (58%):**
The core bottleneck is a **data gap**, not a model failure. The job database contains only mid-to-senior level roles from Deloitte and JPMorgan Chase. When users ask for entry-level positions or apply multi-criteria filters (e.g., *"beginner DA in New York, Python only"*), no matching documents exist in the Vector Search index, and semantic similarity search cannot replicate structured SQL-style filtering.

### Recommended Improvements
1. **Expand the dataset** — add entry-level and junior-level postings to cover beginner queries
2. **Add metadata filtering** — attach structured fields (location, seniority, skills) to index records and use hybrid search to filter before retrieval
3. **Increase token limit** — raise `max_tokens` from 500 to handle multi-result responses without truncation